In [ ]:
import os
os.environ["LANG"] = "sv"
os.environ["PHONEMIZER"] = "espeak"
os.environ["ALPHABET"] = "ipa"
os.environ["BASE_CKPT_URL"] = "https://huggingface.co/OpenVoiceOS/phoonnx_eu-ES_miro_espeak/resolve/main/epoch%3D299-step%3D99600.ckpt"
os.environ["BASE_CONFIG_URL"] = "https://huggingface.co/OpenVoiceOS/phoonnx_eu-ES_miro_espeak/resolve/main/miro_eu-ES.json"
os.environ["HF_DATASET"] = "TigreGotico/tts-train-synthetic-miro_sv-SE"
#os.environ["HF_TOKEN"] = "hf_xxxxx"  # hugging face token, for private datasets and to avoid rate limiting
os.environ["UV_VENV_CLEAR"] = "1"
os.environ["ACCELERATOR"] = 'gpu'  # gpu or cpu
os.environ["CUDA"] = "1"  # tell ByT5 phonemizer to use gpu
os.environ["LOCAL_CKPT_PATH"] = "checkpoints/prev.ckpt"
os.environ["LOCAL_CONFIG_PATH"] = "checkpoints/prev.ckpt.json"
os.environ["LOCAL_DATASET_PATH"] = "/kaggle/working/dataset"  # /kaggle/working/dataset  or /kaggle/input/xxx if you uploaded dataset
os.environ["PHONEMIZED_DATASET_PATH"] = "/kaggle/working/training"  # output of preprocess.py - you may upload it instead of preprocessing in kaggle

In [ ]:
!python3 -m pip install uv
!uv pip install --system wheel 'setuptools==68' 'huggingface_hub[cli]'
!uv venv --python 3.10

In [ ]:
!git clone https://github.com/TigreGotico/phoonnx

In [ ]:
# NOTE: add any extra dependencies here, lang specific phonemizer etc
!uv pip install -e phoonnx[train] --python /kaggle/working/.venv/bin/python3.10

In [ ]:
# NOTE: skip if not using espeak phonemizer
!apt-get install --reinstall -y espeak-ng

In [ ]:
# NOTE: skip this if you uploaded the dataset to kaggle
!huggingface-cli download $HF_DATASET --quiet --repo-type dataset --local-dir $LOCAL_DATASET_PATH

In [ ]:
# NOTE: skip this if you uploaded the checkpoint to kaggle
!wget $BASE_CKPT_URL -O $LOCAL_CKPT_PATH
!wget $BASE_CONFIG_URL -O $LOCAL_CONFIG_PATH

In [ ]:
!source .venv/bin/activate && cd phoonnx/phoonnx_train/vits/monotonic_align && \
mkdir -p monotonic_align && \
cythonize -i core.pyx && \
mv core*.so monotonic_align && ls monotonic_align

In [ ]:
# NOTE: skip this if you uploaded the preprocessed data to kaggle
!source .venv/bin/activate && python phoonnx/phoonnx_train/preprocess.py \
  --language $LANG \
  --input-dir $LOCAL_DATASET_PATH \
  --output-dir $PHONEMIZED_DATASET_PATH \
  --single-speaker \
  --sample-rate 22050 \
  --phoneme-type $PHONEMIZER \
  --alphabet $ALPHABET  \
  --prev-config $LOCAL_CONFIG_PATH  # if finetuning, reutilizes previous phoneme_id_map
 # --add-diacritics # arabic and hebrew
 # --phonemizer-model OpenVoiceOS/g2p-mbyt5-12l-ipa-childes-espeak-onnx  # uncomment if using byt5

In [ ]:
# wait until finished or timeout
!source .venv/bin/activate && python phoonnx/phoonnx_train/train.py \
    --dataset-dir $PHONEMIZED_DATASET_PATH \
    --accelerator $ACCELERATOR \
    --devices 1 \
    --batch-size 16 \
    --validation-split 0.05 \
    --max-epochs 1000 \
    --checkpoint-epochs 1 \
    --precision 32 \
    --resume-from-checkpoint $LOCAL_CKPT_PATH